# LeetCode #417: Pacific Atlantic Water Flow

https://leetcode.com/problems/pacific-atlantic-water-flow/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS from Every Cell)** | $O((m \cdot n)^2)$ | $O(m \cdot n)$ |
| **Optimal: Reverse BFS/DFS from Oceans ★** | $O(m \cdot n)$ | $O(m \cdot n)$ |

---

## Understanding the Methods

### Brute Force (DFS from Every Cell)
For each cell, run DFS/BFS to check if water can reach both the Pacific and Atlantic oceans. This leads to redundant work as many cells are visited multiple times.

### Optimal: Reverse BFS/DFS from Oceans ★
Instead of flowing water downhill from each cell, reverse the direction: start BFS/DFS from the ocean borders and flow uphill (to cells with >= height). Run one pass from Pacific borders and one from Atlantic borders. The answer is the intersection of cells reachable from both.

**Why this is better than Brute Force:** Each cell is visited at most twice (once per ocean), making the total work O(m*n) instead of O((m*n)^2).

**Constraints:**
* m == heights.length, n == heights[i].length
* 1 <= m, n <= 200
* 0 <= heights[i][j] <= 10^5

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<int>> PacificAtlantic(int[][] heights) {
        int m = heights.Length, n = heights[0].Length;
        bool[,] pacific = new bool[m, n], atlantic = new bool[m, n];
        for (int i = 0; i < m; i++) {
            Dfs(heights, pacific, i, 0, m, n);
            Dfs(heights, atlantic, i, n - 1, m, n);
        }
        for (int j = 0; j < n; j++) {
            Dfs(heights, pacific, 0, j, m, n);
            Dfs(heights, atlantic, m - 1, j, m, n);
        }
        var result = new List<IList<int>>();
        for (int i = 0; i < m; i++)
            for (int j = 0; j < n; j++)
                if (pacific[i, j] && atlantic[i, j])
                    result.Add(new List<int> { i, j });
        return result;
    }

    private void Dfs(int[][] h, bool[,] visited, int r, int c, int m, int n) {
        if (visited[r, c]) return;
        visited[r, c] = true;
        int[] dr = {0, 0, 1, -1}, dc = {1, -1, 0, 0};
        for (int d = 0; d < 4; d++) {
            int nr = r + dr[d], nc = c + dc[d];
            if (nr >= 0 && nr < m && nc >= 0 && nc < n && h[nr][nc] >= h[r][c])
                Dfs(h, visited, nr, nc, m, n);
        }
    }
}

### Python

In [ ]:
class Solution:
    def pacificAtlantic(self, heights: list[list[int]]) -> list[list[int]]:
        if not heights:
            return []
        m, n = len(heights), len(heights[0])
        pacific, atlantic = set(), set()

        def dfs(r, c, visited):
            visited.add((r, c))
            for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in visited and heights[nr][nc] >= heights[r][c]:
                    dfs(nr, nc, visited)

        for i in range(m):
            dfs(i, 0, pacific)
            dfs(i, n - 1, atlantic)
        for j in range(n):
            dfs(0, j, pacific)
            dfs(m - 1, j, atlantic)
        return [[r, c] for r, c in pacific & atlantic]

### Go

In [ ]:
func pacificAtlantic(heights [][]int) [][]int {
    m, n := len(heights), len(heights[0])
    pacific := make([][]bool, m)
    atlantic := make([][]bool, m)
    for i := range pacific {
        pacific[i] = make([]bool, n)
        atlantic[i] = make([]bool, n)
    }
    dirs := [][2]int{{0, 1}, {0, -1}, {1, 0}, {-1, 0}}
    var dfs func(int, int, [][]bool)
    dfs = func(r, c int, visited [][]bool) {
        visited[r][c] = true
        for _, d := range dirs {
            nr, nc := r+d[0], c+d[1]
            if nr >= 0 && nr < m && nc >= 0 && nc < n && !visited[nr][nc] && heights[nr][nc] >= heights[r][c] {
                dfs(nr, nc, visited)
            }
        }
    }
    for i := 0; i < m; i++ {
        dfs(i, 0, pacific)
        dfs(i, n-1, atlantic)
    }
    for j := 0; j < n; j++ {
        dfs(0, j, pacific)
        dfs(m-1, j, atlantic)
    }
    var result [][]int
    for i := 0; i < m; i++ {
        for j := 0; j < n; j++ {
            if pacific[i][j] && atlantic[i][j] {
                result = append(result, []int{i, j})
            }
        }
    }
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn pacific_atlantic(heights: Vec<Vec<i32>>) -> Vec<Vec<i32>> {
        let (m, n) = (heights.len(), heights[0].len());
        let mut pacific = vec![vec![false; n]; m];
        let mut atlantic = vec![vec![false; n]; m];
        fn dfs(h: &Vec<Vec<i32>>, vis: &mut Vec<Vec<bool>>, r: usize, c: usize, m: usize, n: usize) {
            vis[r][c] = true;
            for (dr, dc) in [(0i32,1i32),(0,-1),(1,0),(-1,0)] {
                let (nr, nc) = (r as i32 + dr, c as i32 + dc);
                if nr >= 0 && nr < m as i32 && nc >= 0 && nc < n as i32 {
                    let (nr, nc) = (nr as usize, nc as usize);
                    if !vis[nr][nc] && h[nr][nc] >= h[r][c] {
                        dfs(h, vis, nr, nc, m, n);
                    }
                }
            }
        }
        for i in 0..m {
            dfs(&heights, &mut pacific, i, 0, m, n);
            dfs(&heights, &mut atlantic, i, n - 1, m, n);
        }
        for j in 0..n {
            dfs(&heights, &mut pacific, 0, j, m, n);
            dfs(&heights, &mut atlantic, m - 1, j, m, n);
        }
        let mut result = Vec::new();
        for i in 0..m {
            for j in 0..n {
                if pacific[i][j] && atlantic[i][j] {
                    result.push(vec![i as i32, j as i32]);
                }
            }
        }
        result
    }
}

## Example Scenarios

### Scenario 1: Standard island
**Input:** `heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]`  
Cells like (0,4) touch Pacific (top) and water flows down to Atlantic. Multiple cells qualify. **Output:** `[[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]`

### Scenario 2: Single cell
**Input:** `heights = [[1]]`  
Touches both oceans. **Output:** `[[0,0]]`

### Scenario 3: Flat grid
**Input:** `heights = [[1,1],[1,1]]`  
All cells can reach both oceans since water flows freely. **Output:** `[[0,0],[0,1],[1,0],[1,1]]`

### Scenario 4: Single row
**Input:** `heights = [[1,2,3]]`  
Pacific on left, Atlantic on right. Cell (0,2) can reach both: flows left to Pacific and is on Atlantic border. **Output:** `[[0,0],[0,1],[0,2]]`

### Scenario 5: Peak in center
**Input:** `heights = [[1,1,1],[1,9,1],[1,1,1]]`  
The center peak (1,1) can reach both oceans. All border cells also qualify. **Output:** `[[0,0],[0,1],[0,2],[1,0],[1,1],[1,2],[2,0],[2,1],[2,2]]`

*Infographic will be added in a future update.*